In [2]:
# @title
import os
os.environ["GROQ_API_KEY"] = "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"

## Install libraries

In [5]:
!pip install -q youtube-transcript-api langchain-community langchain_groq \
               faiss-cpu tiktoken python-dotenv langchain_huggingface \
               langchain_text_splitters

In [6]:
from langchain_huggingface import HuggingFaceEmbeddings

In [7]:
from youtube_transcript_api import YouTubeTranscriptApi, TranscriptsDisabled
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_groq import ChatGroq
from langchain_community.vectorstores import FAISS
from langchain_core.prompts import PromptTemplate

## Step 1a - Indexing (Document Ingestion)

In [8]:
video_id = "AultJcNb90c" # only the ID, not full URL
from youtube_transcript_api import YouTubeTranscriptApi, TranscriptsDisabled # Import get_transcript directly
    # If you don’t care which language, this returns the “best” one
transcript_list = YouTubeTranscriptApi().fetch(video_id, languages=['en']) # Call get_transcript directly


In [9]:
video_id = "AultJcNb90c" # only the ID, not full URL
from youtube_transcript_api import YouTubeTranscriptApi, TranscriptsDisabled # Import get_transcript directly
try:
    # If you don’t care which language, this returns the “best” one
    transcript_list = YouTubeTranscriptApi().fetch(video_id, languages=['en']) # Call get_transcript directly

    # Flatten it to plain text
    transcript = " ".join(chunk.text for chunk in transcript_list)
    print(transcript)

except TranscriptsDisabled:
    print("No captions available for this video.")

(person 1 speaking in foreign language) (person 2 speaking in foreign language) - This is a video from Douyin,
China's version of TikTok. (person 1 speaking in foreign language) - It's not real. It's a skit; a spoof. That guy on the left is
supposed to be an official from the state Birth Encouragement Office. (person 1 speaking in foreign language) - [Max] And he's yelling
at this guy to tell him, "Hey, you better have kids." (person 1 speaking in foreign language) - [Max] The joke is that
this is what it feels like to be in China nowadays. (person 2 speaking in foreign language) - China has been getting very intense about pushing its citizens to have babies. They're offering subsidies to new parents, cash handouts to people who marry young, free childbirth services, free preschool, even bonus points on school entrance exams for second- and third-born kids. They're even taxing condoms
and birth control pills, so Chinese couples will
have more unprotected sex. This all started two years

## Step 1b - Indexing (Text Splitting)

In [10]:
splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
chunks = splitter.create_documents([transcript])

In [11]:
len(chunks)

33

In [12]:
chunks[5]

Document(metadata={}, page_content='good social safety net, bad social safety net, always happens. (light thoughtful music) Here\'s an Our World in\nData thing that shows this. Each dot is a country. Moving right means it\'s getting richer; down means lower birth rate. There they go, down and\nright, down and right. And if you\'re wondering, "Okay, but why?" The answer has to do\nwith a bunch of things. A big one is people\'s values changing as they have more\nopportunities and options that come with greater\nwealth and education. But it\'s more than that, too. We don\'t need to get into\nall of that right now, but if people want, we could. So let us know in the comments. But for now, just know that\nwhen a country gets richer, people have fewer kids. Okay? Good? Anyway, look at that chart again. Let\'s just highlight a\nfew of the rich countries: the US, Japan, Russia, France, all clustered together, right? But here\'s China, and\nit\'s kind of an outlier. It\'s like it ended up with 

## Step 1c & 1d - Indexing (Embedding Generation and Storing in Vector Store)

In [15]:
embeddings = HuggingFaceEmbeddings(model_name="BAAI/bge-small-en-v1.5")
vector_store = FAISS.from_documents(chunks, embeddings)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/743 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/366 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [18]:
vector_store.index_to_docstore_id

{0: '5c4c2b69-380b-4d23-bf02-a39f6de14292',
 1: '1cd65719-5f54-4289-abbd-19cbc8265694',
 2: 'af58e3f2-1d65-49fb-addc-6663ee642678',
 3: '19fa7554-75f4-4a47-adbd-24226255b8d0',
 4: '726801e7-985c-4b10-8eb3-f83fa3d0e18b',
 5: '1282d5ff-d9f1-44f2-8fe2-070908efd70f',
 6: 'a4ca084a-da66-4942-bebd-98a1f35ca24f',
 7: '3dfd1834-b1ae-4aca-9147-27fc853a6e56',
 8: 'a53f3fe4-2d27-444f-a4b8-f54b9bb9811a',
 9: '5f78d017-d605-455f-a4a6-8d01c2c0fe24',
 10: '6143e1f6-d08b-419c-8670-49e1be613c1c',
 11: 'c8c608f5-19f0-4db4-a7da-31181a2dcb39',
 12: 'ae060d47-9a83-4d97-9eab-9b4a15943296',
 13: '536067ec-5ca8-426c-bdff-4b9bec52bef3',
 14: '1490ff27-e75a-48d6-9c55-7bf09fb0cf4c',
 15: 'ee8dce22-a9e4-4865-868b-4bd4026c837c',
 16: '9c3a4b38-140e-4bd0-991c-e1ac06625584',
 17: 'ede8e14f-95ba-491b-aa95-49f994413bb6',
 18: 'b0b539b5-7aaf-4bfb-8743-51f8c213b730',
 19: '35226634-4860-4c5f-85f0-a84ef86594a9',
 20: 'e62460fe-422f-4436-b219-224fb528a3bd',
 21: '05a62827-1f49-42fe-9d03-516d221652b6',
 22: '8697129d-05e8-

In [19]:
vector_store.get_by_ids(['19fa7554-75f4-4a47-adbd-24226255b8d0'])

[Document(id='19fa7554-75f4-4a47-adbd-24226255b8d0', metadata={}, page_content='China\'s recent history that make it the country it is today. This is a big economic thing, so we\'re gonna build our\nway up to what it is, and why it\'s got Xi going to incredible lengths to reverse it, to the point that Chinese people are joking about government thugs individually bullying\nthem into having babies. Because if China can\'t fix this, and so far it has not figured\nout how, no one has actually, the Chinese century will end\njust as it\'s getting started. (bright lively music) To understand this problem\nlooming over China, we\'ve gotta look at what it\nand the world will look like at the end of this century if they fail, and at this one very alarming chart that shows where things are headed. It\'s a story that all traces back to something called\nthe "one-child policy," something you may have heard of. But let\'s start with a particular problem that basically every country\nencounters soone

## Step 2 - Retrieval

In [20]:
retriever = vector_store.as_retriever(search_type="similarity", search_kwargs={"k": 4})

In [21]:
retriever

VectorStoreRetriever(tags=['FAISS', 'HuggingFaceEmbeddings'], vectorstore=<langchain_community.vectorstores.faiss.FAISS object at 0x7fc7a7113ec0>, search_kwargs={'k': 4})

In [25]:
retriever.invoke('is china pushing more kids?')

[Document(id='5c4c2b69-380b-4d23-bf02-a39f6de14292', metadata={}, page_content='(person 1 speaking in foreign language) (person 2 speaking in foreign language) - This is a video from Douyin,\nChina\'s version of TikTok. (person 1 speaking in foreign language) - It\'s not real. It\'s a skit; a spoof. That guy on the left is\nsupposed to be an official from the state Birth Encouragement Office. (person 1 speaking in foreign language) - [Max] And he\'s yelling\nat this guy to tell him, "Hey, you better have kids." (person 1 speaking in foreign language) - [Max] The joke is that\nthis is what it feels like to be in China nowadays. (person 2 speaking in foreign language) - China has been getting very intense about pushing its citizens to have babies. They\'re offering subsidies to new parents, cash handouts to people who marry young, free childbirth services, free preschool, even bonus points on school entrance exams for second- and third-born kids. They\'re even taxing condoms\nand birth c

## Step 3 - Augmentation

In [37]:
llm = ChatGroq(model="llama-3.1-8b-instant", temperature=0.2)

In [27]:
prompt = PromptTemplate(
    template="""
      You are a helpful assistant.
      Answer ONLY from the provided transcript context.
      If the context is insufficient, just say you don't know.

      {context}
      Question: {question}
    """,
    input_variables = ['context', 'question']
)

In [39]:
question          = "is the topic of new birth policy in china discussed? if yes then what was discussed"
retrieved_docs    = retriever.invoke(question)

In [40]:
retrieved_docs

[Document(id='1cd65719-5f54-4289-abbd-19cbc8265694', metadata={}, page_content='and birth control pills, so Chinese couples will\nhave more unprotected sex. This all started two years ago when Xi Jinping started talking about a new culture of marriage and childbirth. This whole speech was Xi\ntelling government officials it was now a matter of national urgency to increase childbirth rates. Ever since, China\'s state media\nhas been bombarding people with messages about their\npatriotic duty to procreate. Officials are even starting\nto call up young women one by one to say, "Hey, shouldn\'t you be getting\npregnant right now?" So what happened to get\nXi Jinping so spooked? Why is the machinery of the\nbiggest, most centralized state in human history all\nsuddenly shifting into gear for this, for babies? I think I have a pretty good idea. (pulsing thoughtful music) It has to do with the fact\nthat China\'s population, after centuries of going\nup, is starting to decline. And it went do

In [41]:
context_text = "\n\n".join(doc.page_content for doc in retrieved_docs)
context_text

'and birth control pills, so Chinese couples will\nhave more unprotected sex. This all started two years ago when Xi Jinping started talking about a new culture of marriage and childbirth. This whole speech was Xi\ntelling government officials it was now a matter of national urgency to increase childbirth rates. Ever since, China\'s state media\nhas been bombarding people with messages about their\npatriotic duty to procreate. Officials are even starting\nto call up young women one by one to say, "Hey, shouldn\'t you be getting\npregnant right now?" So what happened to get\nXi Jinping so spooked? Why is the machinery of the\nbiggest, most centralized state in human history all\nsuddenly shifting into gear for this, for babies? I think I have a pretty good idea. (pulsing thoughtful music) It has to do with the fact\nthat China\'s population, after centuries of going\nup, is starting to decline. And it went down for\n\n(person 1 speaking in foreign language) (person 2 speaking in foreign

In [42]:
final_prompt = prompt.invoke({"context": context_text, "question": question})

In [43]:
final_prompt

StringPromptValue(text='\n      You are a helpful assistant.\n      Answer ONLY from the provided transcript context.\n      If the context is insufficient, just say you don\'t know.\n\n      and birth control pills, so Chinese couples will\nhave more unprotected sex. This all started two years ago when Xi Jinping started talking about a new culture of marriage and childbirth. This whole speech was Xi\ntelling government officials it was now a matter of national urgency to increase childbirth rates. Ever since, China\'s state media\nhas been bombarding people with messages about their\npatriotic duty to procreate. Officials are even starting\nto call up young women one by one to say, "Hey, shouldn\'t you be getting\npregnant right now?" So what happened to get\nXi Jinping so spooked? Why is the machinery of the\nbiggest, most centralized state in human history all\nsuddenly shifting into gear for this, for babies? I think I have a pretty good idea. (pulsing thoughtful music) It has to 

## Step 4 - Generation

In [44]:
answer = llm.invoke(final_prompt)
print(answer.content)

Yes, the topic of the new birth policy in China is discussed. 

According to the transcript, the new birth policy in China is to increase childbirth rates, as stated by Xi Jinping, the leader of China. The policy aims to encourage couples to have more children, and the government is offering various incentives such as subsidies, cash handouts, free childbirth services, and bonus points on school entrance exams for second- and third-born kids. The government is also taxing condoms and birth control pills to discourage the use of birth control.


## Building a Chain

In [45]:
from langchain_core.runnables import RunnableParallel, RunnablePassthrough, RunnableLambda
from langchain_core.output_parsers import StrOutputParser

In [46]:
def format_docs(retrieved_docs):
  context_text = "\n\n".join(doc.page_content for doc in retrieved_docs)
  return context_text

In [47]:
parallel_chain = RunnableParallel({
    'context': retriever | RunnableLambda(format_docs),
    'question': RunnablePassthrough()
})

In [48]:
parallel_chain.invoke('who is Demis')

{'context': 'giving Anthropic an ultimatum to remove restrictions. But Engadget, classified as "lean left," describes the Pentagon as\nchallenging the company\'s safety guardrails. That\'s a real distinction. It\'s why Ground News is a great way to understand how the\nmedia is shaping stories. Zapier.com calls it one of the\nbest news apps on the market. It\'s completely subscriber-funded. Get the unlimited access\nVantage plan for 40% off by going to ground.news/maxfisher, or scan the QR code on your screen. Now, back to China. (light thoughtful music) So look at this chart again. By 2015, when China ended\nthe one-child policy, yeah, birth rates have been\nreally low for a long time, but remember that below this line, a country\'s population will shrink. China pushed itself so far past this line that it is now triggering something that the architects of this\npolicy never anticipated. Something very, very dangerous. Okay, remember this chart? This is the time bomb. To explain it, I w

In [49]:
parser = StrOutputParser()

In [50]:
main_chain = parallel_chain | prompt | llm | parser

In [51]:
main_chain.invoke('Can you summarize the video')

'The video discusses China\'s population control measures and their impact on the country. It starts with a skit from a Chinese video platform, Douyin, where a man is being yelled at by an official from the state Birth Encouragement Office to have kids. The host, Max, explains that this is a joke about the current situation in China, where the government is pushing citizens to have babies.\n\nThe video then discusses the history of China\'s population control measures, including the one-child policy, which was implemented in 1979. The host mentions a documentary called "One Child Nation" that shows the shocking violence and coercion used to enforce the policy, including forced abortions and sterilizations.\n\nThe host then explains that China\'s population control measures have been successful in reducing the birth rate, but the country is now facing a population crisis. The host uses a chart to illustrate the concept of a "magic number" that indicates when a country\'s population is i